In [33]:
import sys
from pathlib import Path

for src_path in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if src_path.exists():
        sys.path.append(str(src_path))
from data import generate_induction_data
import torch
import transformers as tr
import transformer_lens as tl

In [34]:
gpt2_tokenizer = tr.GPT2Tokenizer.from_pretrained("gpt2")
tensors, metadata = generate_induction_data(tokenizer=gpt2_tokenizer, num_samples=50, return_metadata=True)

In [35]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
gpt2 = tl.HookedTransformer.from_pretrained("gpt2", device=device, dtype=dtype)
gpt2.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

In [36]:
def cache_values_by_suffix(cache_entry, suffix):
    return {
        name: value.detach().cpu()
        for name, value in cache_entry.cache_dict.items()
        if name.endswith(suffix)
    }


attention_patterns = []
hook_z = []
last_token_logits = []

with torch.inference_mode():
    for input_tensor in tensors:
        input_tensor = input_tensor.unsqueeze(0).to(device)

        logits, cache_entry = gpt2.run_with_cache(
            input_tensor,
            names_filter=lambda name: name.endswith("attn.hook_pattern") or name.endswith("attn.hook_z"),
            device="cpu",
        )
        
        attention_patterns.append(cache_values_by_suffix(cache_entry, "attn.hook_pattern"))
        hook_z.append(cache_values_by_suffix(cache_entry, "attn.hook_z"))
        last_token_logits.append(logits[:, -1].detach().cpu())

        del logits, cache_entry, input_tensor

torch.cuda.empty_cache()

In [37]:
from core.induction import cal_induction_scores, cal_logit_attribution_score, display_induction_scores, score_grid

induction_scores = cal_induction_scores(attention_patterns, metadata)
logit_attribution_scores = cal_logit_attribution_score(hook_z, tensors, gpt2, metadata)

In [39]:
top_heads = display_induction_scores(
    induction_scores,
    logit_attribution_scores,
    sort_by="induction_score",
    top_k=20,
)

display(score_grid(induction_scores, metric="induction_score"))

,layer,head,previous_token_score,induction_score,logit_attribution_score
0,5,5,0.038,0.914,4.726
1,6,9,0.035,0.842,15.870
2,5,1,0.033,0.795,9.617
3,7,10,0.036,0.750,14.871
4,10,7,0.043,0.544,-23.293
5,5,0,0.067,0.518,3.695
6,9,6,0.048,0.489,20.996
7,7,2,0.036,0.473,13.025
8,10,1,0.059,0.464,13.213
9,9,9,0.043,0.442,20.135


head,0,1,2,3,4,5,6,7,8,9,10,11
layer,,,,,,,,,,,,
0,0.037348,0.000255,0.027997,0.000243,0.004180,0.000012,0.035400,0.021289,0.034870,0.036747,0.021997,3.539027e-02
1,0.020695,0.027503,0.039452,0.016632,0.033615,0.031677,0.026301,0.038717,0.029784,0.029523,0.043076,3.188657e-03
2,0.019140,0.042891,0.000159,0.006691,0.001916,0.003914,0.030846,0.036012,0.007174,0.003967,0.034741,1.514257e-02
3,0.002396,0.015717,0.001593,0.000373,0.029903,0.033676,0.006163,0.001060,0.002981,0.005267,0.015240,1.155923e-02
4,0.001948,0.004882,0.016965,0.007610,0.006083,0.004104,0.022122,0.024612,0.035862,0.009358,0.007579,4.968452e-10
5,0.518163,0.795388,0.004747,0.035093,0.005029,0.914058,0.000809,0.001015,0.310387,0.048295,0.023719,6.973430e-03
6,0.012644,0.038076,0.022635,0.022328,0.055685,0.021499,0.045939,0.036894,0.001609,0.841852,0.090010,1.751053e-02
7,0.005044,0.228935,0.473430,0.053509,0.003814,0.016983,0.088771,0.101423,0.007301,0.029429,0.750485,1.563416e-01
8,0.018926,0.314315,0.031397,0.052581,0.005214,0.019948,0.257217,0.007278,0.079810,0.036740,0.106579,3.331574e-02
